# Fine-tune the station's YOLO26n on the merged drone dataset — round 5 (ft_v5)

**Before you run:** `Runtime → Change runtime type → GPU (T4)`.

Put these in Google Drive, folder **`MyDrive/drone_training/`**:
- `drone_dataset_v5.zip` — built by `training/build_dataset.py`: v3 plus
  frames from the station's own cameras (`training/station_frames.py`):
  a Mavic flown in front of the station (143 frames, every drone box checked
  by eye) and only the HARD backgrounds — the windows where ft_v2 boxed
  something that is not a drone (wires, a pole lamp, car lights, window
  frames). Round 4 had ~2 960 station backgrounds against 91 drone images and
  lost 8 points of recall (06.10.2026); the easy ones are left out here.
- `best_ft_v2.pt` — the ft_v2 model (`runs/ft_v2/weights/best.pt`, renamed):
  round 5 starts from it, as rounds 3 and 4 did
- `hailo_dataflow_compiler-3.33.1-….whl` — the Hailo Dataflow Compiler wheel
  (the version the current HEF was built with; the Pi runs HailoRT 4.23.0)

Everything the training writes goes straight to Drive (`drone_training/runs/`),
so a Colab disconnect loses nothing — see the **Resume** cell. The HEF part at
the end can run in a later session: it reads everything from Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/drone_training'
DATA_ZIP = f'{DRIVE}/drone_dataset_v5.zip'
BASE_PT = f'{DRIVE}/best_ft_v2.pt'
RUNS = f'{DRIVE}/runs'
RUN_NAME = 'ft_v5'
import os
assert os.path.exists(DATA_ZIP), f'upload {DATA_ZIP}'
assert os.path.exists(BASE_PT), f'upload {BASE_PT}'

Same Ultralytics version the station's models were trained with (8.4.66),
so nothing about the model format changes.

In [ ]:
!pip -q install ultralytics==8.4.66
import ultralytics; ultralytics.checks()

Unzip to the Colab disk (much faster to read than Drive).

In [ ]:
!rm -rf /content/drone_dataset && mkdir -p /content/drone_dataset
!unzip -q "$DATA_ZIP" -d /content/drone_dataset
DATA_YAML = '/content/drone_dataset/data.yaml'
# The dataset root must be the folder of data.yaml. Ultralytics resolves a
# relative 'path:' against the CURRENT directory, so drop any such line.
import re
_y = open(DATA_YAML).read()
open(DATA_YAML, 'w').write(re.sub(r'(?m)^path:.*\n', '', _y))
!cat /content/drone_dataset/data.yaml
!for s in train val; do echo "$s: $(ls /content/drone_dataset/images/$s | wc -l) images"; done
!cat /content/drone_dataset/SOURCES.md

A look at random training images with their boxes — check the labels make sense.

In [ ]:
import random, glob, cv2
import matplotlib.pyplot as plt
imgs = random.sample(glob.glob('/content/drone_dataset/images/train/*.jpg'), 12)
fig, axes = plt.subplots(3, 4, figsize=(20, 14))
for ax, p in zip(axes.flat, imgs):
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    lab = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    for row in open(lab).read().strip().splitlines():
        _, cx, cy, bw, bh = map(float, row.split())
        x0, y0 = int((cx - bw / 2) * w), int((cy - bh / 2) * h)
        cv2.rectangle(im, (x0, y0), (int(x0 + bw * w), int(y0 + bh * h)), (255, 0, 0), max(1, w // 300))
    ax.imshow(im); ax.set_title(p.split('/')[-1][:40], fontsize=8); ax.axis('off')
plt.tight_layout()

## Baseline — the CURRENT model on the NEW validation set
This is the honest starting point. The model's own training report said
mAP50 0.994; at the station it scored 0.04 on a drone. This number is what
the fine-tune has to beat.

In [ ]:
from ultralytics import YOLO
base = YOLO(BASE_PT)
base_metrics = base.val(data=DATA_YAML, imgsz=640, batch=32, plots=False)
print('BASELINE  mAP50 %.3f  mAP50-95 %.3f  P %.3f  R %.3f' % (
    base_metrics.box.map50, base_metrics.box.map, base_metrics.box.mp, base_metrics.box.mr))

## Smoke test — 1 epoch on 5 % of the data (a minute), before the long run

In [ ]:
import os, time
LAST = f'{RUNS}/{RUN_NAME}/weights/last.pt'
if os.path.exists(LAST):
    print('the run already exists - smoke test skipped')
else:
    t0 = time.time()
    YOLO(BASE_PT).train(data=DATA_YAML, epochs=1, imgsz=640, batch=32, fraction=0.05,
                        project='/content/smoke', name='smoke', exist_ok=True, plots=False)
    print(f'smoke test: {(time.time() - t0) / 60:.1f} min')

## Fine-tune from the ft_v2 model
Same image size (640) and architecture (YOLO26n), so the Hailo compile path
does not change. ft_v2 already knows the drone data; round 5 mostly has to
learn the station's scene, so 25 epochs with `patience=8` (one epoch took
~9 min on a Colab T4 on 25.09.2026 with the v2 set).

The BEFORE/AFTER below is on the whole val split. 614 of its 2 009 images
were in ft_v2's own training set (the split moved between rounds), so
BEFORE is ft_v2 partly on images it has seen: the fair comparison is made on
the Pi afterwards, on the 1 395 images neither model has seen.

**Safe to re-run, and after a disconnect just Run all again**: the cell
continues an unfinished run from `last.pt`, skips a finished one, and
starts fresh only when there is no run yet.

In [ ]:
import os, torch
from ultralytics import YOLO
LAST = f'{RUNS}/{RUN_NAME}/weights/last.pt'
# Ultralytics marks a finished run by stripping its checkpoint to epoch -1.
state = 'new'
if os.path.exists(LAST):
    ep = torch.load(LAST, map_location='cpu', weights_only=False).get('epoch', -1)
    state = 'finished' if ep == -1 else f'unfinished (last epoch {ep + 1})'
print('run', RUN_NAME, '->', state)
if state == 'new':
    YOLO(BASE_PT).train(
        data=DATA_YAML, imgsz=640, epochs=25, patience=8, batch=32,
        optimizer='auto', cos_lr=True, close_mosaic=10,
        mosaic=1.0, scale=0.5, translate=0.1, fliplr=0.5,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        project=RUNS, name=RUN_NAME, exist_ok=True, seed=0, workers=2)
elif state.startswith('unfinished'):
    YOLO(LAST).train(resume=True)

## Before / after on the same validation set

In [ ]:
from ultralytics import YOLO
best_pt = f'{RUNS}/{RUN_NAME}/weights/best.pt'
base = YOLO(BASE_PT)                 # again: after a disconnect it is gone
new = YOLO(best_pt)
if 'base_metrics' not in globals():
    base_metrics = base.val(data=DATA_YAML, imgsz=640, batch=32, plots=False)
new_metrics = new.val(data=DATA_YAML, imgsz=640, batch=32, plots=True)
for name, m in (('BEFORE (base model)', base_metrics), ('AFTER  (best.pt)   ', new_metrics)):
    print('%s  mAP50 %.3f  mAP50-95 %.3f  P %.3f  R %.3f' % (name, m.box.map50, m.box.map, m.box.mp, m.box.mr))

## The test that matters — frames from the station
If you recorded frames on the station (`visual.frame_record`), put the
drone frames WITH labels (and backgrounds without) in
`MyDrive/drone_training/station_val/` (`*.jpg` + same-name `*.txt`).
These were never trained on. Scores are per image: the drone's best score,
and the strongest box on background frames (false positives).

In [ ]:
import glob, os
SV = f'{DRIVE}/station_val'
files = sorted(glob.glob(f'{SV}/*.jpg')) if os.path.isdir(SV) else []
print(len(files), 'station frames')
for label, m in (('BEFORE', base), ('AFTER ', new)):
    drone_best, bg_worst = [], []
    for f in files:
        r = m.predict(f, imgsz=640, conf=0.01, verbose=False)[0]
        s = [float(c) for c in r.boxes.conf]
        (drone_best if os.path.exists(f[:-4] + '.txt') else bg_worst).append(max(s) if s else 0.0)
    if drone_best:
        print(label, 'drone frames: median best score %.3f, found at >=0.4 in %d/%d' % (
            sorted(drone_best)[len(drone_best) // 2], sum(x >= 0.4 for x in drone_best), len(drone_best)))
    if bg_worst:
        print(label, 'background frames: false positives >=0.4 in %d/%d' % (
            sum(x >= 0.4 for x in bg_worst), len(bg_worst)))

## Export ONNX for the Hailo Dataflow Compiler
The station's current HEF ends at the six raw head convolutions (box +
class for strides 8/16/32 — see `HEAD_CONFIG` in `TWO_CAMERAS_FIXED.py`).
Compile the new ONNX the same way; the cell prints the candidate end nodes.

In [ ]:
from ultralytics import YOLO
# Written next to best.pt, i.e. on Drive: runs/<RUN_NAME>/weights/best.onnx
onnx_path = YOLO(f'{RUNS}/{RUN_NAME}/weights/best.pt').export(
    format='onnx', imgsz=640, opset=11, simplify=True, dynamic=False)
print(onnx_path)
!pip -q install onnx
import onnx
g = onnx.load(onnx_path).graph
heads = [n.name for n in g.node if n.op_type == 'Conv' and 'one2one' in n.name]
print('last convs of the one2one heads (end nodes for the DFC parse):')
for n in heads[-6:]:
    print('  ', n)

## HEF part — safe to stop and Run all again later
A HEF already on Drive is not compiled again; the steps below are skipped
once it is there. A compile interrupted half-way starts over.

In [ ]:
import glob, os
# The ONNX export above switched this notebook's process to CPU
# (Ultralytics select_device('cpu') sets CUDA_VISIBLE_DEVICES=''), and every
# command started from here would inherit it: the DFC would see no GPU.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
HEF_OUT = f'{RUNS}/{RUN_NAME}/hef'
ONNX = f'{RUNS}/{RUN_NAME}/weights/best.onnx'
TODO = [r for r in ('zoo_fast',) if not glob.glob(f'{HEF_OUT}/*_{r}.hef')]
# A quantized HAR without its HEF: an earlier session optimized and then died
# in the compile. Only the compile is left - no calibration set, no GPU.
HARS = sorted(glob.glob(f'{HEF_OUT}/*_zoo_fast_quantized.har'))
print('HEF still to build:', TODO or 'none - it is on Drive',
      '| compile only, from', HARS[-1].split('/')[-1] if TODO and HARS else '')
sh = get_ipython().system            # what `!` does; f-strings below fill the {…}

## Calibration set — images from the dataset, fed the way the station feeds the chip
1024 images from the **train** split (not val: the optimizer fine-tunes the
quantized weights on them), stratified by drone size, plus the station's own
background frames; letterboxed to 640 with grey 114 like `HailoInference`.

In [ ]:
%%writefile /content/build_calib.py
#!/usr/bin/env python3
"""
build_calib.py — the calibration set for the Hailo compile, taken from the
TRAIN split of the merged dataset and preprocessed exactly the way the
station hands a frame to the HEF.

    python build_calib.py <dataset_dir> <out_dir> [--n 1024] [--seed 0]
    python build_calib.py <dataset_dir> <out_dir> --pick-only   # list only
    python build_calib.py <dataset_dir> <out_dir> --pick calib_pick.txt

<dataset_dir>   the unzipped drone_dataset (images/train, labels/train)
<out_dir>/000000.npy ...   one image per file: float32 (640, 640, 3), RGB, 0..255
<out_dir>/calib_list.txt   which dataset image each file came from, and why
<out_dir>/calib_sheet.jpg  16 of them, for a look

WHY THESE CHOICES
  * TRAIN split only. The optimizer's quantization-aware fine-tune adjusts
    the quantized weights on these images; taking them from val would leak
    val into the model that the comparison is then run on.
  * Letterboxed to 640x640 with grey 114, RGB, values 0..255 — the same
    arithmetic as HailoInference._letterbox in TWO_CAMERAS_FIXED.py (the
    station's 640x480 frame lands between two 80-px grey bars). The chip
    divides by 255 itself (normalization in the model script), so values
    stay 0..255. The current HEF was calibrated on images STRETCHED to
    640x640 (dron_buster/hailo_recompile/build_calib.py) — a geometry the
    station never sends.
  * Stratified by drone size, so each of the three heads sees positives.
    Stride 8 decides the far, small targets; a set of mostly large drones
    would fit its class output's range to background alone. All train
    images without a drone (the station's own frames) are included, up to
    10 % of the set: they are the scene the chip actually looks at.
  * One file per image, not one 5 GB array: compile_hef.py stacks them into
    the in-memory array the current HEF was built from when RAM allows,
    and otherwise hands the Dataflow Compiler the directory itself (its
    "npy_dir" input: one image per .npy file).
"""
import argparse
import random
from pathlib import Path

import cv2
import numpy as np
from PIL import Image

SIZE = 640
PAD = 114
# Long side of the LARGEST drone in the frame, px after letterboxing to 640.
BUCKETS = [("<16", 0, 16, 0.25), ("16-32", 16, 32, 0.30),
           ("32-64", 32, 64, 0.25), (">=64", 64, 1e9, 0.20)]
MAX_BACKGROUND_SHARE = 0.10


def letterbox(img):
    """Copy of HailoInference._letterbox (TWO_CAMERAS_FIXED.py) — keep the
    two identical: calibration must see what the chip is fed."""
    sh, sw = img.shape[:2]
    r = min(SIZE / sh, SIZE / sw)
    nw, nh = int(round(sw * r)), int(round(sh * r))
    left = int(round((SIZE - nw) / 2.0 - 0.1))
    top = int(round((SIZE - nh) / 2.0 - 0.1))
    out = np.full((SIZE, SIZE, 3), PAD, dtype=np.uint8)
    if nw == sw and nh == sh:
        out[top:top + nh, left:left + nw] = img
    else:
        out[top:top + nh, left:left + nw] = cv2.resize(
            img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    return out


def largest_drone_px(label_path, w, h):
    """Long side (px at 640) of the largest box, 0.0 for a background."""
    best = 0.0
    for line in label_path.read_text().splitlines():
        p = line.split()
        if len(p) >= 5:
            best = max(best, max(float(p[3]) * w, float(p[4]) * h))
    return best * SIZE / max(w, h)


def select(ds, n, seed):
    """[(image path, why)] — the calibration images, in the order they go
    into the set (the optimizer's statistics use the first 64)."""
    backgrounds, pools = [], {name: [] for name, *_ in BUCKETS}
    for img in sorted((ds / "images" / "train").glob("*.jpg")):
        lab = ds / "labels" / "train" / (img.stem + ".txt")
        if not lab.exists():
            continue
        with Image.open(img) as im:         # header only, no pixel decode
            w, h = im.size
        px = largest_drone_px(lab, w, h)
        if px == 0.0:
            backgrounds.append(img)
            continue
        for name, lo, hi, _share in BUCKETS:
            if lo <= px < hi:
                pools[name].append(img)
                break

    rng = random.Random(seed)
    n_bg = min(len(backgrounds), int(n * MAX_BACKGROUND_SHARE))
    picked = [(p, "background") for p in rng.sample(backgrounds, n_bg)]
    n_drone = n - n_bg
    for name, _lo, _hi, share in BUCKETS:
        take = min(int(round(n_drone * share)), len(pools[name]))
        rng.shuffle(pools[name])
        picked += [(p, name) for p in pools[name][:take]]
        pools[name] = pools[name][take:]
    # Rounding, or a bucket with too few images: fill from what is left.
    rest = [(p, name) for name, v in pools.items() for p in v]
    rng.shuffle(rest)
    picked += rest[:max(0, n - len(picked))]
    rng.shuffle(picked)
    return picked[:n]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("dataset_dir")
    ap.add_argument("out_dir")
    ap.add_argument("--n", type=int, default=1024)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--pick-only", action="store_true",
                    help="write <out_dir>/calib_pick.txt (the selection, in "
                         "order) and stop — to ship only those images")
    ap.add_argument("--pick", help="a calib_pick.txt to follow instead of "
                                   "selecting: exactly those images, in order")
    args = ap.parse_args()
    ds, out = Path(args.dataset_dir), Path(args.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    if args.pick:
        picked = []
        for line in Path(args.pick).read_text().splitlines():
            why, name = line.split("\t")
            picked.append((ds / "images" / "train" / name, why))
    else:
        picked = select(ds, args.n, args.seed)
    if args.pick_only:
        (out / "calib_pick.txt").write_text(
            "".join(f"{why}\t{p.name}\n" for p, why in picked))
        print(f"{len(picked)} picked -> {out / 'calib_pick.txt'}")
        return
    for old in out.glob("*.npy"):
        old.unlink()

    counts = {}
    with open(out / "calib_list.txt", "w") as f:
        for i, (p, why) in enumerate(picked):
            bgr = cv2.imread(str(p))
            rgb = letterbox(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
            np.save(out / f"{i:06d}.npy", rgb.astype(np.float32))
            f.write(f"{i:06d}.npy\t{why}\t{p.name}\n")
            counts[why] = counts.get(why, 0) + 1
    print(f"calibration set: {len(picked)} images -> {out}")
    print("  by largest drone (px at 640):",
          ", ".join(f"{k} {counts.get(k, 0)}"
                    for k in [b[0] for b in BUCKETS] + ["background"]))

    tiles = [cv2.resize(cv2.cvtColor(np.load(out / f"{i:06d}.npy")
                                     .astype(np.uint8), cv2.COLOR_RGB2BGR),
                        (320, 320)) for i in range(min(16, len(picked)))]
    while len(tiles) < 16:
        tiles.append(np.zeros((320, 320, 3), np.uint8))
    sheet = np.vstack([np.hstack(tiles[r * 4:r * 4 + 4]) for r in range(4)])
    cv2.imwrite(str(out / "calib_sheet.jpg"), sheet)


if __name__ == "__main__":
    main()

In [ ]:
if TODO and not HARS:
    sh('python /content/build_calib.py /content/drone_dataset /content/calib --n 1024')
    import shutil; shutil.copy('/content/calib/calib_list.txt', f'{RUNS}/{RUN_NAME}/')
    from IPython.display import Image as _Img, display
    display(_Img('/content/calib/calib_sheet.jpg', width=900))
else:
    print('calibration not needed:', 'the HEF is on Drive' if not TODO
          else 'the quantized HAR is on Drive, only the compile is left')

## Hailo Dataflow Compiler — its own venv, from the wheel on Drive
The DFC pins its own TensorFlow; it gets a separate Python so it cannot break
Ultralytics. The last line must list a GPU: without one the optimizer skips
the quantization-aware fine-tune the current HEF was built with.

In [ ]:
import glob, re, subprocess, zipfile
from packaging.specifiers import SpecifierSet
wheels = sorted(glob.glob(f'{DRIVE}/hailo_dataflow_compiler-*.whl'))
assert wheels, f'put the DFC wheel (hailo_dataflow_compiler-3.33.1-...whl) in {DRIVE}'
WHEEL = wheels[-1]
DFC_VER = re.search(r'hailo_dataflow_compiler-([0-9.]+)', WHEEL).group(1)
print(WHEEL)
if DFC_VER != '3.33.1':
    print(f'⚠️ DFC {DFC_VER}, not 3.33.1: the Pi runs HailoRT 4.23.0 (the DFC 3.33 '
          'release). A HEF from another DFC version may refuse to load there.')
with zipfile.ZipFile(WHEEL) as z:
    meta = z.read(next(n for n in z.namelist() if n.endswith('.dist-info/METADATA'))).decode()
m = re.search(r'^Requires-Python: (.+)$', meta, re.M)
REQ_PY = m.group(1).strip() if m else ''
PY = next(v for v in ['3.10', '3.11', '3.12', '3.9', '3.8']
          if not REQ_PY or SpecifierSet(REQ_PY).contains(v + '.0'))
print('Requires-Python:', REQ_PY or '(not stated)', '-> venv Python', PY)
VPY = '/content/hailo_venv/bin/python'

In [ ]:
if TODO:
    sh('apt-get -qq update > /dev/null && apt-get -qq install -y graphviz libgraphviz-dev > /dev/null')
    sh('pip -q install uv')
    sh(f'uv venv -q --allow-existing --python {PY} /content/hailo_venv')
    sh(f'uv pip install -q --python {VPY} "{WHEEL}" psutil')
    TF_VER = subprocess.check_output([VPY, '-c', 'import tensorflow as t; print(t.__version__)'],
                                     stderr=subprocess.DEVNULL).decode().strip()
    print('DFC pins TensorFlow', TF_VER, '- adding its CUDA libraries')
    sh(f'uv pip install -q --python {VPY} "tensorflow[and-cuda]=={TF_VER}"')
    check = ("import tensorflow as tf, hailo_sdk_client as h; print('DFC', "
             "getattr(h, '__version__', '?'), '| GPU:', tf.config.list_physical_devices('GPU'))")
    sh(f'{VPY} -c "{check}" 2>&1 | tail -1')
else:
    print('the HEF is on Drive - DFC install skipped')

## Compile the HEF
Recipe **`zoo_fast`** — that of the station's current
`bestty_yolo26_1002_1255_zoo_fast.hef` (Hailo's own yolo26 layout, 16-bit
heads, the default level-2 optimization), so old and new HEF differ only in
the weights and the calibration images. Measured on the Pi on 02.10.2026, it
runs the detector ~12 % faster than the `logits` layout in crop mode.

ft_v2's `zoo_fast` build took ~37 min on a Colab T4 (optimization 17.5 min).
The HEF goes to Drive (`runs/<RUN_NAME>/hef/`) when it is done. The script
stops rather than build a HEF with a weaker optimization than the current one.

**If the session runs out of memory here** (round 4 did, right after
training): `Runtime → Restart session`, then Run all again — the finished
training is skipped and the compile gets the whole RAM.

In [ ]:
%%writefile /content/compile_hef.py
#!/usr/bin/env python3
"""
compile_hef.py — ONNX -> HEF for the station's Hailo-8L with DFC 3.33.1,
the compiler the station's current HEF (bestty_yolo260508.hef) was built
with, so that a difference between HEFs comes from what was changed on
purpose, not from the compiler.

    <dfc venv>/bin/python compile_hef.py --onnx best.onnx --calib <dir> \\
        --out <dir> --recipe logits|sigmoid|zoo_fast|zoo
    <dfc venv>/bin/python compile_hef.py --from-har <...>_zoo_fast_quantized.har \\
        --out <dir> --recipe zoo_fast

--from-har  the compile alone, from the quantized HAR a run saves right
          after the optimization — for a session that died in the compile
          (Colab ran out of memory there after training, ft_v4 and ft_v5).
          Needs no GPU and no calibration set.

--calib   a directory of .npy files, one image each: float32 (640, 640, 3),
          RGB, 0..255, letterboxed like the station (build_calib.py)
--out     receives bestty_yolo26_<stamp>_<recipe>.hef, the quantized .har,
          the DFC logs and compile_summary_<recipe>.json
--recipe  logits   AS THE CURRENT HEF: the class outputs leave the chip as
                   logits and the station applies the sigmoid. Measured on
                   the Pi (25.09.2026): that HEF's class logits are
                   quantized in steps of ~0.69-0.79, so above 0.4 it can
                   only report 0.5 / 0.665 / 0.69 / 0.80 / 0.83 / 0.887 /
                   0.915, never more.
          sigmoid  the sigmoid moves ONTO the chip (conv64/80/94), so what
                   is quantized is 0..1 in steps of ~0.004. NOT USED: its
                   quantization-aware fine-tune diverges on this model —
                   28.09.2026, total distill loss 2.09 -> 6.51 -> 5.57 ->
                   5.34 over the 4 epochs, conv94's pinned at 1.0000,
                   while `logits` fell 1.17 -> 1.13 -> 1.04 -> 1.00; an
                   earlier compile this way (5.08.2026) logged an SNR of
                   -1.1 dB on the same conv94.

          zoo_fast HAILO'S OWN LAYOUT, our optimization. Hailo's model-zoo
                   script for yolo26n/yolo26s on Hailo-8L (hailo_model_zoo
                   v2.18 = DFC 3.33.0 / HailoRT 4.23,
                   cfg/alls/generic/yolo26n.alls) sets NO
                   context_switch_param and keeps the six head convs, their
                   outputs and dw1/dw6/dw7/dw8 in 16 bit. Measured on the
                   station's Pi 5 (hailortcli, batch 1, 02.10.2026): their
                   yolo26n HEF has 4 contexts and runs 47.7 FPS (PCIe Gen 2)
                   / 78.2 FPS (Gen 3), ours (`logits`) 6 contexts, 37.8 /
                   65.2 FPS. Which of the differences makes the 4 contexts is
                   NOT known. The optimization stays the default one (level 2,
                   as `logits`), so this recipe only changes the layout.
          zoo      Hailo's script as it is: on top of zoo_fast, optimization
                   level 4 with AdaRound (calibset 1024, batch 8). The DFC's
                   own docstring calls AdaRound "time consuming and requires
                   strong system" — on Colab's 12.7 GB it may run out of
                   memory; its duration there is not known.

COMMON TO ALL (as the current HEF)
  * model name `bestty_yolo26`: the HEF's outputs are then named
    bestty_yolo26/conv61 ... conv94, which HEAD_CONFIG in
    TWO_CAMERAS_FIXED.py expects (another name and the station falls back
    to shape-based auto-detection);
  * the graph cut at the six one2one head convolutions (box + class for
    strides 8/16/32);
  * normalization([0,0,0],[255,255,255]): the station feeds UINT8 0..255;
  * context_switch_param(mode=enabled) — logits/sigmoid only;
  * the default optimization — level 2 with a GPU and >= 1024 images
    (statistics on 64 images, 4 epochs of fine-tune on 1024) — all but zoo.
  The station's decoder detects either output kind by itself.
  Unlike the current HEF, the layer noise analysis (a report, not a step
  that changes the model) is switched off: it ran out of memory on Colab.

REFUSES rather than quietly producing a worse HEF when
  * an end node or one of the six output layers is not where it should be;
  * TensorFlow in this venv sees no GPU, or there are fewer than 1024
    calibration images: the optimizer would silently drop to level 0 (no
    quantization-aware fine-tune) and say so only in its log.
"""
import argparse
import json
import os
import re
import shutil
import sys
import time
from pathlib import Path

os.environ.setdefault("USER", "colab")        # the DFC reads it
# ⚠️ GPU selection, before the DFC is imported.
#  * Ultralytics' ONNX export calls select_device("cpu"), which sets
#    CUDA_VISIBLE_DEVICES="" in the notebook's process, and every command
#    started from the notebook afterwards inherits it: TensorFlow saw no GPU
#    on a T4 runtime (Colab, 25 and 28.09.2026). An empty or negative value
#    is therefore taken as "not chosen", not as "no GPU".
#  * The DFC's own selector (hailo_model_optimization/__init__.py) runs only
#    when the variable is unset, and rejects a GPU with > 5 % of its memory
#    in use — the notebook's PyTorch holds ~1.4 GB of the T4 — so it is
#    bypassed by naming the device.
if os.environ.get("CUDA_VISIBLE_DEVICES", "").strip() in ("", "-1"):
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"

MODEL_NAME = "bestty_yolo26"
END_NODES = [
    "/model.23/one2one_cv2.0/one2one_cv2.0.2/Conv",
    "/model.23/one2one_cv3.0/one2one_cv3.0.2/Conv",
    "/model.23/one2one_cv2.1/one2one_cv2.1.2/Conv",
    "/model.23/one2one_cv3.1/one2one_cv3.1.2/Conv",
    "/model.23/one2one_cv2.2/one2one_cv2.2.2/Conv",
    "/model.23/one2one_cv3.2/one2one_cv3.2.2/Conv",
]
# (box layer, class layer, channels) per stride — HEAD_CONFIG's names.
HEADS = [("conv61", "conv64", 8), ("conv77", "conv80", 16),
         ("conv91", "conv94", 32)]
NORMALIZATION = (
    "norm_layer = normalization([0.0, 0.0, 0.0], [255.0, 255.0, 255.0])\n")
# The layer noise analysis only REPORTS per-output SNR ("does not modify the
# structure of the model's graph", CheckerConfig in the 3.33.1 wheel). On
# Colab (12.7 GB) it was killed for memory (exit -9) right after the
# fine-tune had finished, twice (28.09.2026) — so it is off.
CHECKER_OFF = "model_optimization_config(checker_cfg, policy=disabled)\n"
MODEL_SCRIPT = (NORMALIZATION + "context_switch_param(mode=enabled)\n"
                + CHECKER_OFF)
# Hailo's 16-bit layers for yolo26n/s (cfg/alls/generic/yolo26n.alls, model
# zoo v2.18), names as in their script. check_layers() refuses to start if
# the parsed model does not have them.
ZOO_16BIT = (
    "quantization_param([dw1, dw6, dw7, dw8], precision_mode=a16_w16)\n"
    "quantization_param([conv61, conv77, conv91, conv64, conv80, conv94], "
    "precision_mode=a16_w16)\n"
    "quantization_param([output_layer1, output_layer2, output_layer3, "
    "output_layer4, output_layer5, output_layer6], precision_mode=a16_w16)\n"
)
RECIPES = {
    "logits": MODEL_SCRIPT,
    "sigmoid": MODEL_SCRIPT + "".join(
        f"change_output_activation({cls}, sigmoid)\n" for _, cls, _ in HEADS),
    "zoo_fast": NORMALIZATION + ZOO_16BIT + CHECKER_OFF,
    "zoo": (NORMALIZATION
            + "model_optimization_config(calibration, batch_size=8, "
              "calibset_size=1024)\n"
            + "model_optimization_flavor(optimization_level=4, "
              "compression_level=0)\n"
            + "post_quantization_optimization(adaround, policy=enabled, "
              "batch_size=8)\n"
            + ZOO_16BIT + CHECKER_OFF),
}
MIN_CALIB = 1024
IMG_BYTES = 640 * 640 * 3 * 4


def check_onnx(path):
    import onnx
    names = {n.name for n in onnx.load(path).graph.node}
    missing = [n for n in END_NODES if n not in names]
    if missing:
        heads = sorted(n for n in names if "one2one" in n and n.endswith("Conv"))
        sys.exit("end nodes not in the ONNX: %s\none2one convs present:\n  %s"
                 % (missing, "\n  ".join(heads)))


def check_heads(runner):
    """The six output layers must be HEAD_CONFIG's convs, box = 4 channels,
    class = 1 channel, at 80/40/20 cells. (Checked against the HN of the
    current model's HAR: output_layer1..6 <- conv61, 64, 77, 80, 91, 94.)"""
    layers = runner.get_hn_dict()["layers"]
    feeding = set()
    for layer in layers.values():
        if layer.get("type") == "output_layer":
            feeding.update(layer.get("input", []))
    problems = []
    for box, cls, stride in HEADS:
        cells = 640 // stride
        for short, ch in ((box, 4), (cls, 1)):
            full = f"{MODEL_NAME}/{short}"
            if full not in layers:
                problems.append(f"{full}: no such layer")
                continue
            if feeding and full not in feeding:
                problems.append(f"{full}: not an output")
            shape = (layers[full].get("output_shapes") or [[None]])[0]
            if list(shape[-3:]) != [cells, cells, ch]:
                problems.append(f"{full}: shape {shape}, expected "
                                f"[.., {cells}, {cells}, {ch}]")
    if problems:
        sys.exit("parsed model does not match HEAD_CONFIG:\n  "
                 + "\n  ".join(problems))
    print("heads OK: " + ", ".join(f"{b}/{c} (stride {s})"
                                   for b, c, s in HEADS))


def check_layers(runner, recipe):
    """Every layer a quantization_param names must exist in the parsed
    model — checked BEFORE the optimizer runs, so a name that differs from
    Hailo's yolo26n costs seconds, not an hour of Colab."""
    script = "".join(l for l in RECIPES[recipe].splitlines(True)
                     if l.startswith("quantization_param"))
    wanted = sorted(set(re.findall(r"\b(?:dw|conv|output_layer)\d+\b",
                                   script)))
    if not wanted:
        return
    layers = runner.get_hn_dict()["layers"]
    missing = [n for n in wanted if f"{MODEL_NAME}/{n}" not in layers]
    if missing:
        dws = sorted((n.split("/")[-1] for n in layers
                      if n.split("/")[-1].startswith("dw")),
                     key=lambda s: int(s[2:]) if s[2:].isdigit() else 0)
        sys.exit(f"recipe {recipe}: layers not in the parsed model: "
                 f"{missing}\n depthwise layers present: {dws}")
    print(f"16-bit layers OK: {', '.join(wanted)}")


def build_runner(onnx_path, recipe):
    from hailo_sdk_client import ClientRunner
    runner = ClientRunner(hw_arch="hailo8l")
    runner.translate_onnx_model(onnx_path, MODEL_NAME,
                                start_node_names=["images"],
                                end_node_names=END_NODES)
    check_heads(runner)
    check_layers(runner, recipe)
    runner.load_model_script(RECIPES[recipe])
    return runner


def calib_strategies(files):
    """Ways to hand the calibration set to runner.optimize, most proven
    first. Both are read by the DFC one image at a time through a
    generator (hailo_model_optimization/acceleras/utils/dataset_util.py in
    the 3.33.1 wheel), so the array costs its own size in RAM, not twice."""
    import numpy as np
    import psutil
    avail = psutil.virtual_memory().available
    # The array's 5 GB plus room for the optimizer and the compiler: with
    # only 4 GB of room the process was killed for memory on Colab.
    need = len(files) * IMG_BYTES + (10 << 30)

    def as_array():                 # what built the current HEF
        a = np.empty((len(files), 640, 640, 3), np.float32)
        for i, f in enumerate(files):
            a[i] = np.load(f)
        return a

    def as_dir():                   # DFC "npy_dir": one image per file,
        return str(files[0].parent)     # memory-mapped as it is read

    if avail >= need:
        return [("in-memory array", as_array), ("npy directory", as_dir)]
    print(f"RAM available {avail / 2**30:.1f} GB < {need / 2**30:.1f} GB "
          "for the in-memory array: the directory first")
    return [("npy directory", as_dir), ("in-memory array", as_array)]


def log_facts(text):
    """What the DFC's client log says about the optimization it ran."""
    lines = text.splitlines()
    return {
        "optimization_level": [l for l in lines
                               if "optimization level" in l.lower()][:5],
        "optimization_level_reduced": [l for l in lines
                                       if "Reducing optimization level" in l],
        "qft_lines": [l for l in lines if "for finetune" in l],
        "snr_db": dict(re.findall(r"(output_layer\d+) SNR:\s*(-?[\d.]+) dB",
                                  text)),
    }


def save_logs(out, recipe):
    for f in Path(".").glob("*.log"):
        shutil.copy(f, out / f"{recipe}_{f.name}")


def show_har(har):
    """A HAR is a tar archive: list it, and quote whatever its text members
    say about the optimization — the HAR is all that is left when the
    session that optimized it died before the compile finished."""
    import tarfile
    keys = re.compile(r"optimization_level|flavor|finetune|qft|Reducing|"
                      r"calibset|sdk_version|state", re.I)
    with tarfile.open(har) as t:
        for m in t.getmembers():
            print(f"  {m.name}  {m.size} B")
            if m.isfile() and m.size < (4 << 20) and not m.name.endswith(".npz"):
                text = t.extractfile(m).read().decode("utf-8", "replace")
                hits = [l.strip()[:200] for l in text.splitlines() if keys.search(l)]
                for l in hits[:12]:
                    print("     ", l)


def compile_from_har(args, out, dfc):
    """The compile alone, from a quantized HAR an earlier run saved: no GPU,
    no calibration set, no optimization — the weights are already the ones
    the optimizer produced. Only for recipes whose model script has no
    compile-time command (context_switch_param is one), so that compiling
    the HAR is the same as the compile that run would have done."""
    har = Path(args.from_har).resolve()
    if "context_switch_param" in RECIPES[args.recipe]:
        sys.exit(f"--from-har: recipe {args.recipe} has a compile-time command "
                 "that a HAR may not carry; optimize again instead")
    if not har.name.endswith(f"_{args.recipe}_quantized.har"):
        sys.exit(f"{har.name} is not a {args.recipe} quantized HAR")
    print("HAR", har)
    show_har(har)
    from hailo_sdk_client import ClientRunner
    runner = ClientRunner(har=str(har))
    check_heads(runner)
    t0 = time.time()
    hef = out / (har.name[:-len("_quantized.har")] + ".hef")
    hef.write_bytes(runner.compile())
    # The optimization's own log, if the run that wrote the HAR saved it.
    opt_log = out / f"{args.recipe}_hailo_sdk.client.log"
    facts = (log_facts(opt_log.read_text(errors="replace")) if opt_log.exists()
             else {"optimization_level": "NOT KNOWN: the log of the run that "
                   "optimized this HAR was not saved"})
    summary = {"hef": hef.name, "recipe": args.recipe, "dfc": dfc,
               "compiled_from_har": har.name,
               "model_script": RECIPES[args.recipe], **facts,
               "compile_min": round((time.time() - t0) / 60, 1)}
    (out / f"compile_summary_{args.recipe}.json").write_text(
        json.dumps(summary, indent=1))
    for f in Path(".").glob("*.log"):
        shutil.copy(f, out / f"{args.recipe}_compile_{f.name}")
    print(json.dumps(summary, indent=1))
    print(f"\nSUCCESS: {hef}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--onnx")
    ap.add_argument("--calib")
    ap.add_argument("--out", required=True)
    ap.add_argument("--recipe", choices=sorted(RECIPES), required=True)
    ap.add_argument("--work", help="scratch folder for the DFC (default: "
                    "/content/hef_work_<recipe> on Colab, else under --out)")
    ap.add_argument("--from-har", help="compile this <...>_<recipe>_quantized"
                    ".har (saved by a run whose compile did not finish) "
                    "instead of optimizing again; needs no GPU")
    args = ap.parse_args()
    if not args.from_har and not (args.onnx and args.calib):
        ap.error("--onnx and --calib are required unless --from-har")
    out = Path(args.out).resolve()
    out.mkdir(parents=True, exist_ok=True)
    work = Path(args.work) if args.work else (
        (Path("/content") if Path("/content").exists() else out)
        / f"hef_work_{args.recipe}")
    shutil.rmtree(work, ignore_errors=True)   # its logs must be this run's
    work.mkdir(parents=True)
    os.chdir(work)                      # the DFC writes its logs to the cwd

    import hailo_sdk_client
    import tensorflow as tf
    dfc = getattr(hailo_sdk_client, "__version__", "?")
    print("DFC", dfc, "| TensorFlow", tf.__version__)
    if args.from_har:
        compile_from_har(args, out, dfc)
        return
    onnx_path = str(Path(args.onnx).resolve())
    if not tf.config.list_physical_devices("GPU"):
        sys.exit("TensorFlow in this venv sees no GPU: the optimizer would "
                 "drop to level 0 (no quantization-aware fine-tune). Fix the "
                 "CUDA install first.")
    files = sorted(Path(args.calib).glob("*.npy"))
    if len(files) < MIN_CALIB:
        sys.exit(f"{len(files)} calibration images < {MIN_CALIB}: the "
                 "optimizer would drop to a lower level")
    check_onnx(onnx_path)

    t0 = time.time()
    runner, used, errors = None, None, []
    for label, make in calib_strategies(files):
        runner = build_runner(onnx_path, args.recipe)
        try:
            print(f"optimize: calibration as {label}", flush=True)
            runner.optimize(make())
            used = label
            break
        except Exception as exc:          # try the next way of feeding it
            errors.append(f"{label}: {type(exc).__name__}: {exc}")
            print("  failed —", errors[-1][:500], flush=True)
    if used is None:
        sys.exit("optimize failed every way:\n  " + "\n  ".join(errors))
    t_opt = time.time() - t0

    stamp = time.strftime("%m%d_%H%M")
    base = f"{MODEL_NAME}_{stamp}_{args.recipe}"
    runner.save_har(str(out / f"{base}_quantized.har"))
    # The optimization log goes next to the HAR NOW: a session that dies in
    # the compile (ft_v4 and ft_v5 did, out of memory) leaves the HAR, and
    # without this log nothing says at which level it was optimized.
    save_logs(out, args.recipe)
    hef = out / f"{base}.hef"
    hef.write_bytes(runner.compile())
    t_all = time.time() - t0

    log = Path("hailo_sdk.client.log")
    text = log.read_text(errors="replace") if log.exists() else ""
    summary = {
        "hef": hef.name, "recipe": args.recipe,
        "model_script": RECIPES[args.recipe], "dfc": dfc, "onnx": onnx_path,
        "calibration_images": len(files), "calibration_as": used,
        "failed_ways": errors, **log_facts(text),
        "optimize_min": round(t_opt / 60, 1),
        "total_min": round(t_all / 60, 1),
    }
    (out / f"compile_summary_{args.recipe}.json").write_text(
        json.dumps(summary, indent=1))
    save_logs(out, args.recipe)
    print(json.dumps(summary, indent=1))
    if summary["optimization_level_reduced"]:
        print("\n⚠️  THE OPTIMIZER REDUCED ITS LEVEL — this HEF is not built "
              "the way the current one was. Do not deploy it.")
    print(f"\nSUCCESS: {hef}")


if __name__ == "__main__":
    main()

In [ ]:
if 'zoo_fast' in TODO and HARS:
    sh(f'{VPY} /content/compile_hef.py --recipe zoo_fast --from-har "{HARS[-1]}" '
       f'--out "{HEF_OUT}"')
elif 'zoo_fast' in TODO:
    sh(f'{VPY} /content/compile_hef.py --recipe zoo_fast --onnx "{ONNX}" '
       f'--calib /content/calib --out "{HEF_OUT}"')
else:
    print('zoo_fast HEF already on Drive - skipped')
sh(f'ls -la "{HEF_OUT}"')

## Next
Download from `MyDrive/drone_training/runs/<RUN_NAME>/`:
`hef/bestty_yolo26_<stamp>_zoo_fast.hef`, `hef/compile_summary_zoo_fast.json`
and `weights/best.pt` → `D:/d/DocumentsFromC/GitHub/drone_training/new_model_v5/`.
The HEF is then compared on the Pi with the current one on the same images
(`training/compare_hef.py`), and live on the station.